<a href="https://colab.research.google.com/github/kostya27111984-ai/Simulative/blob/main/Sim_case_1_10_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2 кейс

**Соберите данные из Google Sheets и напишите функцию `generate_report`, которая принимает три списка (данные с каждого листа в Google Sheets) и сохраняет отчет в файл `student_debt_report.txt`**

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [19]:
!wget https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json

!wget https://gist.github.com/Vs8th/39c5deed0f5539d781f00328f7fd4fe0/raw/result.txt

--2026-04-29 19:36:19--  https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.github.com (gist.github.com)... 140.82.116.3
Connecting to gist.github.com (gist.github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json [following]
--2026-04-29 19:36:19--  https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2358 (2.3K) [text/plain]
Saving to: ‘creds.json.1’

creds.json.1        100%[===================>]   2.30K  --.-KB/s    in 0s      

2026-04-29 19:36:19 (38.9 MB/s) - ‘creds.json.1’ save

Чтобы посмотреть как выглядят сами гугл таблицы в оригинале - можете перейти по [ссылке](https://docs.google.com/spreadsheets/d/1hRnw-PEftF0J-6KY7InlILVwWdkJu4vJiGwRjuE_P4U/edit?usp=sharing) и изучить.

Небольшое описание столбцов в таблицах:  
**Лист1:** `student_id` - айди студента; `student_name` - имя студента; `installment` (Y/N) - факт наличия рассрочки у студента (Y - рассрочка есть, N - рассрочки нет).  
**Лист2:** `student_id` - айди студента; `last_payment_date` - дата последней полученной оплаты; `expected_payment_date` - дата, в которую ожидаем следующий платеж (рассчитывается от даты последнего платежа).  
**Лист3:** `student_id` - айди студента; `already_payed_amount` - уже выплаченная часть рассрочки; `left_to_pay` - оставшаяся (невыплаченная) часть; `one-time_payment` - единоразовый платеж; `installment_amount` - общая сумма, которая бралась в рассрочку.

Как примерно должен выглядеть результат:

In [ ]:
with open('result.txt') as f:
  print(f.read())

Студент Иванов У.У. - долг 100000 рублей
Студент Петров Е.Е. - долг 250000 рублей
Студент Сидоров И.И. - долг 10000 рублей


In [ ]:
#@title Если возникнут трудности с определением `scope`, подсказка:

scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

**Примечание**

Считать должников необходимо на 1 марта 2023 года. То есть определяя количество просроченных платежей, мы определяем их количество не на настоящую дату, а на **1 марта 2023 года**. А периоды внесения платежей для всех студентов одинаковы - **6 месяцев** (183 дня).

То есть, если ожидаемый платеж должен был пройти 3 февраля 2022 года, то к 1 марта 2023 студент просрочил уже три платежа (3 февраля 2022, 3 августа 2022 и 3 февраля 2023). При расчетах отталкивайтесь от даты ожидаемого платежа, разницу между платежами берите 183 дня.

**Решение**

Напишите свое решение ниже

In [ ]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

def generate_report(sheet1, sheet2, sheet3):

  ...



In [20]:
#собрать информацию из Google Sheets
import gspread
from oauth2client.service_account import ServiceAccountCredentials

scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

# Указываем необходимые права доступа к таблицам
scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

# Загружаем ключи аутентификации из файла json
creds = ServiceAccountCredentials.from_json_keyfile_name('creds.json', scope)

# Авторизуемся в Google Sheets API
client = gspread.authorize(creds)

sheet1 = client.open("Installments").worksheet("Лист1")
sheet1_data = sheet1.get_all_records()

sheet2 = client.open("Installments").worksheet("Лист2")
sheet2_data = sheet2.get_all_records()

sheet3 = client.open("Installments").worksheet("Лист3")
sheet3_data = sheet3.get_all_records()




In [23]:
#из первой таблицы выбираем, где рассрочка = Y
s = []
for i in sheet1_data:
  if i['installment'] == 'Y':
    s.append(i)


In [24]:
#обработать ее
#Функция, чтобы 3 слепить списка со словарями по id первого

from typing import List, Dict, Any

def merge_lists_by_id(*lists: List[Dict[str, Any]], key: str = "student_id") -> List[Dict[str, Any]]:
    if not lists:
        return []

    # 1. строим индексы для всех списков, кроме первого
    indices = []
    for lst in lists[1:]:
        idx = {d[key]: d for d in lst}
        indices.append(idx)

    base = lists[0]
    result = []

    for item in base:
        k = item[key]
        merged = item.copy()
        for idx in indices:
            if k in idx:
                # значения из поздних словарей перезапишут поля при конфликте
                merged |= idx[k]          # Python 3.9+ [web:7][web:10]
        result.append(merged)

    return result







In [25]:

#Объединеяем таблицы
res = merge_lists_by_id(s, sheet2_data, sheet3_data, key="student_id")
res

#сгенерировать отчет, в котором будет указано, у каких студентов есть долг
#в текстовый файл формата "Студент Иванов И.И. - долг N рублей"
#обязательно учтите, что иногда встречаются злостные неплательщики, которые пропустили уже несколько платежей
#- это тоже нужно обязательно отразить в отчете. То есть их долг должен быть больше, чем простой разовый платеж.

[{'student_id': 1,
  'student_name': 'Смирнова И.И.',
  'installment': 'Y',
  'last_payment_date': '20.09.2022',
  'expected_payment_date': '19.03.2023',
  'already_payed_amount': 122373,
  'left_to_pay': 277627,
  'one-time_payment': 133333,
  'installment_amount': 400000},
 {'student_id': 2,
  'student_name': 'Кузнецова К.А.',
  'installment': 'Y',
  'last_payment_date': '09.08.2022',
  'expected_payment_date': '05.02.2023',
  'already_payed_amount': 325610,
  'left_to_pay': 474390,
  'one-time_payment': 266666,
  'installment_amount': 800000},
 {'student_id': 3,
  'student_name': 'Петров К.А.',
  'installment': 'Y',
  'last_payment_date': '15.02.2022',
  'expected_payment_date': '14.08.2022',
  'already_payed_amount': 51843,
  'left_to_pay': 148157,
  'one-time_payment': 33333,
  'installment_amount': 200000},
 {'student_id': 4,
  'student_name': 'Смирнова А.А.',
  'installment': 'Y',
  'last_payment_date': '22.12.2022',
  'expected_payment_date': '20.06.2023',
  'already_payed_amou

In [33]:
# из второй таблицы выбираем, что  01.03.2023 > expected_payment_date
# Просроченных платежей = =ОКРУГЛВНИЗ(([@[curren_date]]-[@[last_payment_date]])/182;0)
# Остаток по долгу =ЕСЛИ([@[Просроченных платежей]]*[@[one-time_payment]]>=[@[left_to_pay]];[@[left_to_pay]];[@[Просроченных платежей]]*[@[one-time_payment]])

from datetime import datetime

fmt = '%d.%m.%Y'  # день.месяц.год

final_res = []

for i in res:
  if datetime.strptime('01.03.2023', fmt)  > datetime.strptime(i['expected_payment_date'], fmt):
      number_of_payments = (datetime.strptime('01.03.2023', fmt)  - datetime.strptime(i['last_payment_date'], fmt)).days//182
      #final_res.append({'student_name': i['student_name'], 'loan' : number_of_payments*i['one-time_payment'] if number_of_payments*i['one-time_payment'] < i['left_to_pay'] else i['left_to_pay']})
      final_res.append(f"Студент {i['student_name']} - долг {number_of_payments*i['one-time_payment'] if number_of_payments*i['one-time_payment'] < i['left_to_pay'] else i['left_to_pay']} рублей")
with open("student_debt_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(final_res) + "\n")


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt

import pandas as pd

user_answer = pd.read_csv('student_debt_report.txt')
correct_answer = pd.read_csv('student_debt.txt')

--2026-04-28 18:26:23--  https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.github.com (gist.github.com)... 140.82.112.4
Connecting to gist.github.com (gist.github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt [following]
--2026-04-28 18:26:23--  https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11325 (11K) [text/plain]
Saving to: ‘student_debt.txt’

student_debt.txt    100%[===================>]  11.06K  --.-KB/s    in 0s      

2026-04-28 18:26:23 (94.9 MB/s)

FileNotFoundError: [Errno 2] No such file or directory: 'student_debt_report.txt'

In [ ]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')